# UTS SOAL 1 — Kafka & Wikimedia SSE (25 poin)
**Kernel: BigDataLab, jalan di JupyterLab Master.** Tiap bagian dipisah per sel. Pertahankan data topic yang telah ada; **jangan restart Kafka / membuat producer ganda**. Konfigurasi terbaru dari screenshot: Master `192.168.1.32`, topic `wiki.recentchange.raw`.

**Penting:** angka pada output hanya muncul sesudah sel dieksekusi di cluster; kami tidak memiliki akses menjalankan Kafka privat kelompok.

## 1A — Arsitektur & alasan desain
```text
Wikimedia EventStreams (SSE/JSON)
             │ Python Producer (User-Agent, reconnect, HMAC user anonim)
             ▼
   Kafka Master 192.168.1.32:9092 [wiki.recentchange.raw, 3 partisi]
             │                   │
        Spark RDD/SQL (batch)   Spark Structured Streaming
             └─────────┬─────────┘
                     Hasil dashboard / SQLite (Master)
```

| Parameter | Pilihan | Alasan |
|---|---|---|
| Topic | `wiki.recentchange.raw` | Dipakai semua soal |
| Partisi | 3 (usulan; cek metadata aktual) | Membagi beban konsumsi |
| Key | `wiki:id` | Penyebaran hash lebih rata daripada `wiki` saja; urutan per halaman **tidak** dijamin |
| Replikasi | 1 (broker Master tunggal) | RF>1 memerlukan broker tambahan |
| Retensi | 7 hari (untuk topic baru; cek topic lama) | Cukup untuk percobaan, tetap dipantau kapasitas |
| Consumer group | `uts_s1_decoupling_*` (pengujian khusus) | Offset tersendiri, tidak mengganggu Soal 4 |

**Data skew:** bila key hanya `wiki`, edit dari wiki dominan menumpuk pada satu partisi. Dengan `wiki:id` penyebaran antar edit biasanya lebih baik; tetap ukur distribusi offset aktual.

### SEL 1 — Konfigurasi (jalankan dulu)

In [1]:
# WAJIB DIJALANKAN PERTAMA SETIAP KALI KERNEL DIRESTART. Tidak perlu config.py terpisah.
import os, sys, json, time, socket, datetime, importlib.util
from pathlib import Path
MASTER_IP = os.environ.get('BIGDATA_MASTER_IP', '192.168.1.32')  # terbaru: panel MASTER
KAFKA_BOOTSTRAP = os.environ.get('BIGDATA_KAFKA', MASTER_IP + ':9092')
KAFKA_TOPIC = os.environ.get('BIGDATA_TOPIC', 'wiki.recentchange.raw')
SPARK_MASTER = os.environ.get('BIGDATA_SPARK', 'spark://' + MASTER_IP + ':7077')
# Hasil/output disimpan pada komputer MASTER tempat kernel Jupyter berjalan.
DEFAULT_DIR = 'C:/BigData/work/UTS_Wikipedia_2026' if os.name == 'nt' else str(Path.cwd() / '_hasil_uts')
BASE = Path(os.environ.get('BIGDATA_OUTPUT_DIR', DEFAULT_DIR))
DATA, HASIL, STATE = BASE/'data', BASE/'hasil', BASE/'state'
for folder in (DATA,HASIL,STATE):folder.mkdir(parents=True,exist_ok=True)
MANIFEST_PATH = DATA/'snapshot_offsets.json'
print('KERNEL:',sys.executable)
print('HOST KERNEL:',socket.gethostname())
print('MASTER:',SPARK_MASTER,'| KAFKA:',KAFKA_BOOTSTRAP,'| TOPIC:',KAFKA_TOPIC)
print('FOLDER HASIL:',BASE)


KERNEL: C:\BigData\venv\Scripts\python.exe
HOST KERNEL: BIGDATA-MASTER
MASTER: spark://192.168.1.32:7077 | KAFKA: 192.168.1.32:9092 | TOPIC: wiki.recentchange.raw
FOLDER HASIL: C:\BigData\work\UTS_Wikipedia_2026


### SEL 2 — Cek library/port

In [2]:
# CEK PERPUSTAKAAN DAN KONEKSI. Jangan otomatis memasang ulang paket bawaan BigDataLab.
needed = {'kafka':'kafka-python','requests':'requests'}
missing = [package for module,package in needed.items() if importlib.util.find_spec(module) is None]
if missing:
    raise RuntimeError('Paket Python belum tersedia: '+', '.join(missing)+
      '. Jalankan DI KERNEL BigDataLab: %pip install '+ ' '.join(missing))
from kafka import KafkaConsumer, KafkaAdminClient, TopicPartition, KafkaProducer
from kafka.admin import NewTopic
from kafka.errors import TopicAlreadyExistsError
for port,label in [(7077,'Spark Master'),(9092,'Kafka')]:
    try:
        with socket.create_connection((MASTER_IP,port),timeout=4):
            print(label,port,'TCP TERJANGKAU (bukan jaminan metadata/broker valid).')
    except OSError as exc:
        print(label,port,'TCP GAGAL:',exc)
print('LIBRARY KAFKA TERSEDIA; langkah berikutnya mengecek metadata broker.')


Spark Master 7077 TCP TERJANGKAU (bukan jaminan metadata/broker valid).
Kafka 9092 TCP TERJANGKAU (bukan jaminan metadata/broker valid).
LIBRARY KAFKA TERSEDIA; langkah berikutnya mengecek metadata broker.


### SEL 3 — Soal 1A: metadata topic, buat hanya bila benar-benar belum ada, dan offset (MANDIRI)

In [3]:
# 1A MANDIRI: memastikan variabel terdefinisi, walau sel konfigurasi sempat belum dijalankan.
import os, json
from kafka import KafkaAdminClient, KafkaConsumer, TopicPartition
from kafka.admin import NewTopic
KAFKA_BOOTSTRAP = os.environ.get('BIGDATA_KAFKA','192.168.1.32:9092')
KAFKA_TOPIC = os.environ.get('BIGDATA_TOPIC','wiki.recentchange.raw')
# False karena kelompok sudah memiliki topic dan producer; cegah topic baru akibat salah nama.
CREATE_IF_MISSING = False
admin=KafkaAdminClient(bootstrap_servers=[KAFKA_BOOTSTRAP],client_id='uts_s1_admin',request_timeout_ms=15000)
try:
    topics=admin.list_topics()
    if KAFKA_TOPIC not in topics:
        if not CREATE_IF_MISSING:
            raise RuntimeError('Topic '+KAFKA_TOPIC+' belum ada. Cek nama topic ke Master! '
                'Jangan membuat topic kosong baru tanpa koordinasi. Jika memang BARU, ubah CREATE_IF_MISSING=True.')
        print('MEMBUAT TOPIC BARU 3 PARTISI, RF=1...')
        admin.create_topics([NewTopic(name=KAFKA_TOPIC,num_partitions=3,
             replication_factor=1,topic_configs={'retention.ms':'604800000'})])
    else:print('TOPIC SUDAH ADA; TIDAK DIUBAH:',KAFKA_TOPIC)
finally:admin.close()
consumer=KafkaConsumer(bootstrap_servers=[KAFKA_BOOTSTRAP],group_id=None,enable_auto_commit=False)
try:
    parts=sorted(consumer.partitions_for_topic(KAFKA_TOPIC) or [])
    if not parts:raise RuntimeError('Metadata Kafka kosong; periksa advertised.listeners Master.')
    tps=[TopicPartition(KAFKA_TOPIC,p) for p in parts]
    starts=consumer.beginning_offsets(tps);ends=consumer.end_offsets(tps)
    counts=[]
    print('PARTISI | EARLIEST | LATEST (next offset) | RENTANG')
    for tp in tps:
        row=(tp.partition,starts[tp],ends[tp],ends[tp]-starts[tp]);counts.append(row)
        print(*row,sep=' | ')
    print('JUMLAH RENTANG PESAN TERSEDIA:',sum(v[3] for v in counts))
    print('CATAT: rentang offset bukan bukti pesan unik; topic compacted bisa punya lubang.')
finally:consumer.close()


TOPIC SUDAH ADA; TIDAK DIUBAH: wiki.recentchange.raw


PARTISI | EARLIEST | LATEST (next offset) | RENTANG
0 | 0 | 121140 | 121140
1 | 0 | 108101 | 108101
2 | 0 | 113414 | 113414
3 | 0 | 111319 | 111319
JUMLAH RENTANG PESAN TERSEDIA: 453974
CATAT: rentang offset bukan bukti pesan unik; topic compacted bisa punya lubang.


### SEL 4 — 1B: producer Python SSE minimal 600 detik (**opsional — JANGAN jalankan jika Master sudah punya producer**)

In [10]:
# Ubah RUN_PRODUCER=True HANYA bila kalian ingin mengumpulkan data baru ≥600 detik.
# Jangan jalankan producer kedua secara tidak sengaja.
RUN_PRODUCER=True
EMAIL_KONTAK='stefanyskn@students.amikom.ac.id'  
import hashlib,hmac,ipaddress,secrets,requests
from kafka import KafkaProducer

def private_user(value,key):
    user=str(value or '')
    try:ipaddress.ip_address(user)
    except ValueError:return user
    return 'anon_'+hmac.new(key,user.encode('utf-8'),hashlib.sha256).hexdigest()[:24]

def normalize_event(evt,key):
    if not isinstance(evt,dict) or evt.get('id') is None or not evt.get('wiki'):return None
    sz=evt.get('length') or {}
    return {'id':evt.get('id'),'type':evt.get('type'),'wiki':evt.get('wiki'),
      'server_name':evt.get('server_name'),'title':evt.get('title'),
      'namespace':evt.get('namespace'),'user':private_user(evt.get('user'),key),
      'bot':evt.get('bot'),'minor':evt.get('minor'),'timestamp':evt.get('timestamp'),
      'length':{'old':sz.get('old'),'new':sz.get('new')},'comment':evt.get('comment')}

def parse_sse_lines(lines):
    event_id=None;items=[]
    for raw in lines:
        line=raw.decode('utf-8','replace') if isinstance(raw,bytes) else raw
        if line.startswith('id:'):event_id=line[3:].strip()
        elif line.startswith('data:'):items.append(line[5:].lstrip())
        elif not line:
            if items:yield event_id,'\n'.join(items)
            event_id=None;items=[]

def run_producer_600s():
    if not EMAIL_KONTAK or '@' not in EMAIL_KONTAK:raise ValueError('Isi EMAIL_KONTAK valid.')
    key_file=STATE/'anon_hmac_secret.hex'
    if not key_file.exists():key_file.write_text(secrets.token_hex(32),encoding='ascii')
    private_key=bytes.fromhex(key_file.read_text(encoding='ascii').strip())
    kp=KafkaProducer(bootstrap_servers=[KAFKA_BOOTSTRAP],acks='all',retries=5,
         max_in_flight_requests_per_connection=1,key_serializer=lambda x:x.encode('utf-8'),
         value_serializer=lambda x:json.dumps(x,ensure_ascii=False).encode('utf-8'))
    counters={'queued':0,'ack_ok':0,'ack_error':0,'skipped':0,'reconnect':0,'bad_json':0}
    def ack_ok(_):counters['ack_ok']+=1
    def ack_bad(err):counters['ack_error']+=1;print('ACK ERROR:',str(err)[:120])
    begin=time.monotonic();last_event_id=None
    try:
        while time.monotonic()-begin<600:
            hdr={'User-Agent':f'AMIKOM-ST153-Wiki-UTS/1.0 ({EMAIL_KONTAK})','Accept':'text/event-stream'}
            if last_event_id:hdr['Last-Event-ID']=last_event_id
            try:
                with requests.get('https://stream.wikimedia.org/v2/stream/recentchange',
                    headers=hdr,stream=True,timeout=(15,70)) as resp:
                    resp.raise_for_status()
                    for ev_id,payload in parse_sse_lines(resp.iter_lines()):
                        if time.monotonic()-begin>=600:break
                        try:raw=json.loads(payload)
                        except json.JSONDecodeError:counters['bad_json']+=1;continue
                        clean=normalize_event(raw,private_key)
                        if clean is None:counters['skipped']+=1;continue
                        kp.send(KAFKA_TOPIC,key=f"{clean['wiki']}:{clean['id']}",value=clean).add_callback(ack_ok).add_errback(ack_bad)
                        counters['queued']+=1
                        if ev_id:last_event_id=ev_id
                        if counters['queued']%1000==0:
                            kp.flush(timeout=60)
                            print('ACK:',counters['ack_ok'],'durasi:',round(time.monotonic()-begin,1))
            except (requests.RequestException,ConnectionError,ValueError) as exc:
                counters['reconnect']+=1;print('SSE reconnect:',repr(exc));kp.flush(timeout=30);time.sleep(3)
    finally:kp.flush(timeout=90);kp.close()
    counters['duration_seconds']=round(time.monotonic()-begin,2)
    counters['ack_per_second']=round(counters['ack_ok']/counters['duration_seconds'],2)
    (HASIL/'soal1_producer.json').write_text(json.dumps(counters,indent=2),encoding='utf-8')
    print('HASIL NYATA PRODUCER:',json.dumps(counters,indent=2))
    return counters
if RUN_PRODUCER:
    hasil_prod=run_producer_600s()
else:print('PRODUCER TIDAK DIJALANKAN (aman jika sudah ada data dan bukti dari Master).')


C:\Users\ASUS\AppData\Local\Temp\ipykernel_13276\2655865645.py:74: DeprecationWarning: key_serializer does not implement kafka.serializer.Serializer
  hasil_prod=run_producer_600s()
C:\Users\ASUS\AppData\Local\Temp\ipykernel_13276\2655865645.py:74: DeprecationWarning: value_serializer does not implement kafka.serializer.Serializer
  hasil_prod=run_producer_600s()


ACK: 1000 durasi: 31.9


ACK: 2000 durasi: 47.1


ACK: 3000 durasi: 69.1


ACK: 4000 durasi: 88.2


ACK: 5000 durasi: 111.5


... (18 outputs left out)

ACK: 24000 durasi: 543.6


ACK: 25000 durasi: 566.2


ACK: 26000 durasi: 584.9


HASIL NYATA PRODUCER: {
  "queued": 26647,
  "ack_ok": 26647,
  "ack_error": 0,
  "skipped": 724,
  "reconnect": 0,
  "bad_json": 0,
  "duration_seconds": 600.03,
  "ack_per_second": 44.41
}


### SEL 5 — 1B: ukur laju Kafka dari offset selama 30 detik dan rekam hasilnya

In [11]:
from kafka import KafkaConsumer,TopicPartition
mon=KafkaConsumer(bootstrap_servers=[KAFKA_BOOTSTRAP],group_id=None,enable_auto_commit=False)
try:
    tps=[TopicPartition(KAFKA_TOPIC,p) for p in sorted(mon.partitions_for_topic(KAFKA_TOPIC) or [])]
    if not tps:raise RuntimeError('Topic tidak terlihat dari Kafka.')
    old=mon.end_offsets(tps);t0=time.monotonic();time.sleep(30)
    new=mon.end_offsets(tps);dt=time.monotonic()-t0
    diffs={str(tp.partition):new[tp]-old[tp] for tp in tps}
    report={'interval_seconds':dt,'per_partition':diffs,'messages_added':sum(diffs.values()),
            'rate_messages_sec':sum(diffs.values())/dt if dt else 0}
    print('HASIL KAFKA:',json.dumps(report,indent=2))
    (HASIL/'soal1_rate.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
finally:mon.close()


HASIL KAFKA: {
  "interval_seconds": 30.00361850002082,
  "per_partition": {
    "0": 0,
    "1": 0,
    "2": 0,
    "3": 0
  },
  "messages_added": 0,
  "rate_messages_sec": 0.0
}


### SEL 6 — Bekukan snapshot offset untuk Soal 2 & 3 (jangan timpa manifest lama)

In [12]:
# SNAPSHOT BERSAMA 2 & 3: rentang offset Kafka dibekukan SATU KALI.
# Menyimpan offset, bukan menyalin data privat ke GitHub.
from kafka import KafkaConsumer, TopicPartition
monitor = KafkaConsumer(bootstrap_servers=[KAFKA_BOOTSTRAP],group_id=None,
                        enable_auto_commit=False,request_timeout_ms=30000)
try:
    if MANIFEST_PATH.exists():
        manifest=json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
        print('SNAPSHOT LAMA DIPAKAI TANPA DIUBAH:',MANIFEST_PATH)
    else:
        parts=sorted(monitor.partitions_for_topic(KAFKA_TOPIC) or [])
        if not parts:raise RuntimeError('TOPIC Kafka belum ditemukan: '+KAFKA_TOPIC)
        tps=[TopicPartition(KAFKA_TOPIC,p) for p in parts]
        begins=monitor.beginning_offsets(tps);ends=monitor.end_offsets(tps)
        manifest={'topic':KAFKA_TOPIC,'master_ip_saat_dibekukan':MASTER_IP,
          'dibuat_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
          'start':{str(tp.partition):int(begins[tp]) for tp in tps},
          'end':{str(tp.partition):int(ends[tp]) for tp in tps}}
        manifest['total_offsets']=sum(manifest['end'][p]-manifest['start'][p] for p in manifest['end'])
        if manifest['total_offsets']<=0:raise RuntimeError('Topic belum berisi data untuk snapshot.')
        MANIFEST_PATH.write_text(json.dumps(manifest,indent=2),encoding='utf-8')
        print('SNAPSHOT BARU BEKU:',MANIFEST_PATH)
    if manifest['topic']!=KAFKA_TOPIC:
        raise RuntimeError('Topic pada snapshot berbeda! Periksa '+str(MANIFEST_PATH))
    print('PESAN DLM RENTANG OFFSET:',manifest['total_offsets'])
    print('OFFSET AWAL:',manifest['start'],'OFFSET AKHIR:',manifest['end'])
    if manifest['total_offsets']<200000:
        print('UTS 2: 200.000 ATAU seluruh data terkumpul. Jelaskan jumlah aktual dalam laporan.')
finally:monitor.close()


SNAPSHOT LAMA DIPAKAI TANPA DIUBAH: C:\BigData\work\UTS_Wikipedia_2026\data\snapshot_offsets.json
PESAN DLM RENTANG OFFSET: 453974
OFFSET AWAL: {'0': 0, '1': 0, '2': 0, '3': 0} OFFSET AKHIR: {'0': 121140, '1': 108101, '2': 113414, '3': 111319}


### SEL 7 — 1B: keluaran perintah Kafka CLI (bukti wajib jika file .bat tersedia)

In [13]:
import subprocess
binfolder=Path('C:/kafka/bin/windows')
for label,exe,args in [
   ('INFO TOPIC','kafka-topics.bat',['--describe','--topic',KAFKA_TOPIC]),
   ('EARLIEST','kafka-get-offsets.bat',['--topic',KAFKA_TOPIC,'--time','-2']),
   ('LATEST','kafka-get-offsets.bat',['--topic',KAFKA_TOPIC,'--time','-1'])]:
    path=binfolder/exe
    if path.exists():
        print('===',label,'===')
        res=subprocess.run([str(path),'--bootstrap-server',KAFKA_BOOTSTRAP,*args],
              capture_output=True,text=True,timeout=60)
        print((res.stdout or res.stderr)[-3500:]);print('RETURN CODE:',res.returncode)
    else:print('CLI tidak ditemukan:',path,'(jalankan dari direktori Kafka milik Master bila perlu)')


=== INFO TOPIC ===


Topic: wiki.recentchange.raw	TopicId: wY7M0fzeRWeq068jzH2Txw	PartitionCount: 4	ReplicationFactor: 1	Configs: min.insync.replicas=1,segment.bytes=1073741824,retention.ms=604800000
	Topic: wiki.recentchange.raw	Partition: 0	Leader: 1	Replicas: 1	Isr: 1	Elr: 	LastKnownElr: 
	Topic: wiki.recentchange.raw	Partition: 1	Leader: 1	Replicas: 1	Isr: 1	Elr: 	LastKnownElr: 
	Topic: wiki.recentchange.raw	Partition: 2	Leader: 1	Replicas: 1	Isr: 1	Elr: 	LastKnownElr: 
	Topic: wiki.recentchange.raw	Partition: 3	Leader: 1	Replicas: 1	Isr: 1	Elr: 	LastKnownElr: 

RETURN CODE: 0
=== EARLIEST ===


wiki.recentchange.raw:0:0
wiki.recentchange.raw:1:0
wiki.recentchange.raw:2:0
wiki.recentchange.raw:3:0

RETURN CODE: 0
=== LATEST ===


wiki.recentchange.raw:0:127833
wiki.recentchange.raw:1:114825
wiki.recentchange.raw:2:119959
wiki.recentchange.raw:3:118004

RETURN CODE: 0


### SEL 8 — 1C: simulasi stop consumer 120 detik, offset commit, resume, audit (producer harus aktif)

In [15]:
# UJI TERPISAH; jalankan ketika producer aktif. Tes ini butuh >120 detik.
RUN_DECOUPLING=True
from kafka import KafkaConsumer, TopicPartition
import uuid,collections

def test_decoupling_2menit():
    group='uts_s1_pause_'+uuid.uuid4().hex[:8]
    def create():
        return KafkaConsumer(KAFKA_TOPIC,bootstrap_servers=[KAFKA_BOOTSTRAP],
           group_id=group,enable_auto_commit=False,auto_offset_reset='latest')
    first=create();initial=[]
    try:
        deadline=time.monotonic()+20
        while time.monotonic()<deadline:
            for batch in first.poll(timeout_ms=1000,max_records=1000).values():
                initial.extend(batch)
        first.commit()
        committed={tp.partition:first.committed(tp) for tp in first.assignment()}
        print('CONSUMER 1 STOP | dibaca:',len(initial),'| next committed:',committed)
    finally:first.close()
    mon=KafkaConsumer(bootstrap_servers=[KAFKA_BOOTSTRAP],group_id=None,enable_auto_commit=False)
    try:
        tps=[TopicPartition(KAFKA_TOPIC,p) for p in sorted(mon.partitions_for_topic(KAFKA_TOPIC) or [])]
        before=mon.end_offsets(tps)
        print('CONSUMER BERHENTI 120 DETIK — Kafka/producer tetap berjalan')
        time.sleep(120)
        after=mon.end_offsets(tps)
        increments={tp.partition:after[tp]-before[tp] for tp in tps}
        target={tp.partition:after[tp] for tp in tps}
    finally:mon.close()
    print('JUMLAH PESAN MASUK SAAT BERHENTI (offset):',increments)
    if not sum(increments.values()):print('PERINGATAN: producer tidak menghasilkan pesan saat off; bukti kurang kuat.')
    resumed=create();seen=collections.defaultdict(list);identities=[]
    try:
        deadline=time.monotonic()+120
        # Terima semua data sampai offset target yang dicatat tepat setelah 120 detik.
        while time.monotonic()<deadline:
            batches=resumed.poll(timeout_ms=1000,max_records=3000)
            for group_records in batches.values():
                for record in group_records:
                    seen[record.partition].append(record.offset)
                    try:
                        payload=json.loads(record.value.decode('utf-8'))
                        identities.append((payload.get('wiki'),payload.get('id')))
                    except Exception:pass
            positions={tp.partition:resumed.position(tp) for tp in resumed.assignment()}
            if positions and all(positions.get(p,0)>=limit for p,limit in target.items()):break
        first_after={p:min(offsets) for p,offsets in seen.items() if offsets}
        last_after={p:max(offsets) for p,offsets in seen.items() if offsets}
        replay={p:o for p,o in first_after.items()
          if committed.get(p) is not None and o<committed[p]}
        observed=collections.Counter()
        gaps=[];duplicate_offsets=0
        for p,offs in seen.items():
            ordered=sorted(offs)
            duplicate_offsets+=len(ordered)-len(set(ordered))
            gaps.extend([(p,a,b) for a,b in zip(ordered,ordered[1:]) if b>a+1])
        finished={p:(last_after.get(p,-1)+1)>=limit for p,limit in target.items()}
        resumed.commit()
        summary={'group_id':group,'committed_before':committed,'added_during_stop':increments,
          'target_end_after_120s':target,'first_resumed_offset':first_after,
          'last_resumed_offset':last_after,'replayed_offsets':replay,
          'offset_gaps_inside_received':gaps[:20],'repeated_same_kafka_offset':duplicate_offsets,
          'repeated_logical_wiki_id':len(identities)-len(set(identities)),
          'caught_up_to_pause_target':finished}
        print(json.dumps(summary,indent=2,default=str))
        if not all(finished.values()):print('UJI BELUM TUNTAS: backlog belum habis; tidak boleh klaim 0 pesan hilang.')
        (HASIL/'soal1_decoupling.json').write_text(json.dumps(summary,indent=2,default=str),encoding='utf-8')
        return summary
    finally:resumed.close()
if RUN_DECOUPLING:hasil_1c=test_decoupling_2menit()
else:print('1C tidak otomatis dijalankan. Ubah RUN_DECOUPLING=True dengan producer aktif.')


CONSUMER 1 STOP | dibaca: 0 | next committed: {3: 118004, 0: 127833, 1: 114825, 2: 119959}
CONSUMER BERHENTI 120 DETIK — Kafka/producer tetap berjalan


JUMLAH PESAN MASUK SAAT BERHENTI (offset): {0: 0, 1: 0, 2: 0, 3: 0}
PERINGATAN: producer tidak menghasilkan pesan saat off; bukti kurang kuat.


{
  "group_id": "uts_s1_pause_0193d94d",
  "committed_before": {
    "3": 118004,
    "0": 127833,
    "1": 114825,
    "2": 119959
  },
  "added_during_stop": {
    "0": 0,
    "1": 0,
    "2": 0,
    "3": 0
  },
  "target_end_after_120s": {
    "0": 127833,
    "1": 114825,
    "2": 119959,
    "3": 118004
  },
  "first_resumed_offset": {},
  "last_resumed_offset": {},
  "replayed_offsets": {},
  "offset_gaps_inside_received": [],
  "repeated_same_kafka_offset": 0,
  "repeated_logical_wiki_id": 0,
  "caught_up_to_pause_target": {
    "0": false,
    "1": false,
    "2": false,
    "3": false
  }
}
UJI BELUM TUNTAS: backlog belum habis; tidak boleh klaim 0 pesan hilang.


### Bukti akhir Soal 1
- Screenshot diagram + tabel keputusan desain; metadata topic, jumlah pesan per partisi dan offset Kafka CLI.
- Bukti producer minimal 600 detik (jika arsip lama digunakan, cocokkan **topic** dan broker lama vs baru; jangan mengklaim data lama berada pada topic baru).
- Soal 1C harus ada bukti 120 detik consumer berhenti, offset ter-commit, restart, dan lag/duplikat; cek group Kafka. Tidak mengklaim 0 pesan hilang hanya dari offset berbatas.